# Automated Beamline Alignment at ALS 5.3.1
khchan@lbl.gov, xchong@lbl.gov, awojdyla@lbl.gov; Nov 2025

We would like to deploy the latest version of blop at ALS 5.3.1

We want to show that we can switch between a real beamline (EPICS over ophyd) and a virtual beamline (shadow over ophyd)

one issue is tha shadow doesn't install on python>3.9, so we'll have to make a server.

blop 0.8.1
Following https://nsls-ii.github.io/blop/



In [ ]:
# environment ideally would be als_bl531
import blop 



# files of interest are here:
#~/.ipython/profile_blop/startup


[WARNING 11-26 10:28:29] ax.service.utils.with_db_settings_base: Ax currently requires a sqlalchemy version below 2.0. This will be addressed in a future release. Disabling SQL storage in Ax for now, if you would like to use SQL storage please install Ax with mysql extras via `pip install ax-platform[mysql]`.


## Configure blop

### define degrees of freedom (20-motors)

In [ ]:
import ophyd
from ophyd import EpicsMotor
ophyd.set_cl('caproto')

m101_pitch = EpicsMotor('bl531_esp300:m101_pitch_mm', name='m101_pitch')
#m101_bend  = EpicsMotor('bl531_esp300:m101_bend_um', name='m101_bend')
mono_height = EpicsMotor('bl531_xps1:mono_height_mm', name='mono_height')
mono_angle = EpicsMotor('bl531_xps1:mono_angle_deg', name='mono_angle')

2025-11-26 11:17:41.579 INFO: connection state changed to connected.
2025-11-26 11:17:41.582 INFO: connection state changed to connected.
2025-11-26 11:17:41.584 INFO: connection state changed to connected.
2025-11-26 11:17:41.586 INFO: connection state changed to connected.
2025-11-26 11:17:41.587 INFO: connection state changed to connected.
2025-11-26 11:17:41.588 INFO: connection state changed to connected.
2025-11-26 11:17:41.591 INFO: connection state changed to connected.
2025-11-26 11:17:41.593 INFO: connection state changed to connected.
2025-11-26 11:17:41.595 INFO: connection state changed to connected.
2025-11-26 11:17:41.597 INFO: connection state changed to connected.
2025-11-26 11:17:41.598 INFO: connection state changed to connected.
2025-11-26 11:17:41.611 INFO: connection state changed to connected.
2025-11-26 11:17:41.613 INFO: connection state changed to connected.
2025-11-26 11:17:41.615 INFO: connection state changed to connected.
2025-11-26 11:17:41.617 INFO: conn

### define detectors (21-detectors)

In [4]:
beamstop = ophyd.EpicsSignal('bl201-beamstop:current', name='beamstop')

# author: tmorris@bnl.gov
# June 2023
# as used at ALS

# from datetime import datetime
# import matplotlib.pyplot as plt

# import bluesky.plans as bp
# from bluesky.callbacks import best_effort
# from bluesky.run_engine import RunEngine
# from databroker import Broker
# from ophyd.utils import make_dir_tree

# import ophyd_basler
# from ophyd_basler.basler_handler import BaslerCamHDF5Handler
# from ophyd_basler.basler_camera import BaslerCamera

# ophyd_basler.available_devices()

# #cam num sets the camera we use 
# basler_cam = BaslerCamera(cam_num=3, verbose=True, name="basler_cam")
# basler_cam.exposure_time.put(400)

# import bluesky
# from bluesky import RunEngine
# RE = RunEngine({})

# db = Broker.named("temp")
# db.reg.register_handler("BASLER_CAM_HDF5", BaslerCamHDF5Handler, overwrite=True)


# bec = best_effort.BestEffortCallback()
# bec.disable_plots()
# RE.subscribe(bec)
# RE.subscribe(db.insert)

# root_dir = "/tmp/basler"
# _ = make_dir_tree(datetime.now().year, base_path=root_dir)

### define degrees of freedom (20-motors)

In [5]:
from blop import DOF

dofs = [
    DOF(name="mono_angle", search_domain=(22,23)),
]

# 'm101_pitch'
# 'm101_bend'
# 'mono_height'
# 'mono_angle'

/tmp/ipykernel_4124714/2053054343.py:4: DeprecationWarning: The 'name' argument is deprecated and will be removed in Blop v1.0.0. The `movable.name` will be used instead.
  DOF(name="mono_angle", search_domain=(22,23)),


### define objective function

In [ ]:
import bloptools
import scipy as sp

def get_beam_stats(im):

    fim = sp.ndimage.median_filter(im, size=5)

    threshold = 0.5 * (fim.min() + fim.max())

    mask = fim > threshold

    ny, nx = fim.shape

    x, y = np.arange(nx), np.arange(ny)
    X, Y = np.meshgrid(x, y)

    x_mean = np.sum(mask * X) / mask.sum()
    y_mean = np.sum(mask * Y) / mask.sum()

    x_width =  np.sqrt(np.sum(mask * np.square(X - x_mean)) / mask.sum())
    y_width =  np.sqrt(np.sum(mask * np.square(Y - y_mean)) / mask.sum())

    return fim.max() - fim.min(), x_mean, y_mean, x_width, y_width

In [ ]:
from blop import Objective

objectives = [Objective(name="beamsize", description="rms beam size", target="min")]

### define dataprocessing (metrics/"digestion")

In [ ]:
from ophyd import Device, EpicsSignal, Component as Cpt

class BeamStopCurrent(Device):
    prec = Cpt(EpicsSignal, ".PREC")
    current = Cpt(EpicsSignal, "")

beamstop_current_device = BeamStopCurrent("bl201-beamstop:current", name="beamstop")
beamstop_current_device.prec.put(3)  # This is needed for LiveTable (via BestEffortCallback) to print the values with proper number of decimal values
beamstop_current = beamstop_current_device.current
beamstop_current.kind = "hinted"


# import ophyd
# from PIL import Image
# import scipy as sp
# import epics
# import time as ttime


# class SAXSDetector(Device):
#     image = Cpt(Signal, kind="normal")
#     profile = Cpt(Signal, kind="normal")
#     filename = Cpt(Signal, kind="normal")

#     def __init__(
#         self,
#         center,
#         *args,
#         **kwargs,
#     ):
#         """
#         A class to instantiate a Basler ophyd object.
#         """
#         super().__init__(*args, **kwargs)

#         self.center = center

#     def trigger(self):

#         epics.caput("13PIL1:cam1:Acquire", 1)

#         ttime.sleep(3)

#         filename_ascii = epics.caget('13PIL1:cam1:FullFileName_RBV')
#         self.filename.put(bytes(filename_ascii).decode()[:-1])

#         image = np.array(Image.open(self.filename.get())).clip(max=1e3)
#         self.image.put(image)

#         ny, nx = image.shape
#         x, y = np.arange(nx), np.arange(ny)
#         X, Y = np.meshgrid(x, y)
#         Z = (X - self.center[0]) + 1j * (Y - self.center[1])
#         R = np.abs(Z)

#         bins = np.linspace(0, 500, 256)

#         mask = image > -1

#         profile = sp.stats.binned_statistic(R[mask], image[mask], statistic="mean", bins=bins).statistic

#         self.profile.put(profile)

#         super().trigger()

#         return NullStatus()

#     def stage(self):
#         super().stage()
        
#     def unstage(self):
#         super().unstage()


In [ ]:
def current_digestion(db, uid):

    table = db[uid].table(fill=True)

    product_keys = ["current"]
    products = {key: [] for key in product_keys}

    for row, entry in table.iterrows():

        bad = False

        if bad:
            [products[key].append(np.nan) for key in product_keys]
            continue

        products["current"].append(entry.beamstop)

    return products


#### from 93-digestion

In [ ]:
# from https://nsls-ii.github.io/blop/tutorials/introduction.html
def digestion(df):
    for index, entry in df.iterrows():
        df.loc[index, "beamsize"] = toroid(entry.x_rot, entry.y_rot)

    return df

#### from 93-optimization

In [ ]:
# from bloptools.bayesian import Agent, DOF, Objective




# dofs = [
#     DOF(device=m101_pitch, description="toroid pitch", units="mm", limits=(0, 1)),
#     DOF(device=m101_bend, description="toroid bend",  units="mm", limits=(0, 1)),
#     DOF(device=mono_height, description="monochromator height", units="mm", limits=(0, 1)),
# ]


# objectives = [
#     Objective(key="beam_max", description="beam flux", target="max", log=True),
#     Objective(key="beam_fwhm_x", description="beam width", target="min", log=True),
#     Objective(key="beam_fwhm_y", description="beam height", target="min", log=True),
# ]

# agent = Agent(
#     dofs=dofs,
#     objectives=objectives,
#     digestion=beam_digestion,
#     db=db,
# )


## Start runengine and blop agent

In [ ]:
#from blop.utils import prepare_re_env
# %run -i $prepare_re_env.__file__ --db-type=temp
# -> look here: /home/bl531/miniconda3/envs/blop_shadow/lib/python3.1/site-packages/blop/utils

RE = RunEngine({})
bec = best_effort.BestEffortCallback()
RE.subscribe(bec)

# db = Broker.named(db_type)
# try:
#     databroker.assets.utils.install_sentinels(db.reg.config, version=1)
# except Exception:
#     pass
# RE.subscribe(db.insert)


from databroker import Broker
db = Broker.named('temp')

# Insert all metadata/data captured into db.
RE.subscribe(db.insert)


ImportError: cannot import name 'prepare_re_env' from 'blop.utils' (/home/bl531/miniconda3/envs/als_bl531/lib/python3.12/site-packages/blop/utils/__init__.py)

In [ ]:
from blop import Agent

agent = Agent(
    dofs=dofs,
    objectives=objectives,
    digestion=digestion,
    db=db,
)

### initialize the agent

In [ ]:
RE(agent.learn("quasi-random", n=32))


In [ ]:
agent.plot_objectives()

In [ ]:
# agent.all_acq_funcs
agent.all_acqfs #new

In [ ]:
#agent.plot_acquisition(acq_func="qei")
agent.plot_acquisition(acqf="qei") # new

In [ ]:
agent.ask("qei", n=1)

In [ ]:
# res = agent.ask("qei", n=8, route=True)
# agent.plot_acquisition(acq_func="qei")
# plt.scatter(*res["points"].T, marker="d", facecolor="w", edgecolor="k")
# plt.plot(
#     *res["points"].T,
#     color="r",
# )
# new
res = agent.ask("qei", n=8, route=True)
agent.plot_acquisition(acqf="qei")
plt.scatter(res["points"]["x_rot"], res["points"]["y_rot"], marker="d", facecolor="w", edgecolor="k")
plt.plot(res["points"]["x_rot"], res["points"]["y_rot"], color="r")
plt.show()

In [ ]:
RE(agent.learn("qei", n=4, iterations=8))

In [ ]:
agent.plot_objectives()
print(agent.best)